
# Thí nghiệm 03 — Financial: Huấn luyện và Grid Search

## 1. Mục tiêu

Notebook định nghĩa và quản lý quá trình huấn luyện bốn mô hình dự báo chuỗi thời gian Financial:

- LSTM.
- DeepKoopFormer-PatchTST.
- DeepKoopFormer-Autoformer.
- DeepKoopFormer-Informer.

Dữ liệu đầu vào gồm hai segment của Bitcoin:

- `BTC_segment_1`
- `BTC_segment_2`

Dữ liệu được chuẩn bị trong `01_financial_data.ipynb`. Kiến trúc và các hàm dùng chung được định nghĩa trong `02_financial_model.ipynb`.

## 1.1. Phạm vi

Notebook chứa:

1. Nạp các định nghĩa mô hình bằng AST.
2. Đọc và xác minh dữ liệu Train/Test.
3. Định nghĩa factory khởi tạo mô hình.
4. Định nghĩa hàm huấn luyện một cấu hình.
5. Định nghĩa quy trình Grid Search và checkpointing.
6. Kiểm toán kết quả đã lưu.

## 1.2. Nguyên tắc bảo toàn

Thực nghiệm đã có 224 checkpoint, tương ứng với 224 cấu hình Grid Search và 448 dòng metrics.

Dữ liệu và toàn bộ checkpoint đã được xác minh có cùng SHA-256 fingerprint:

`9383a90ba5109d569c2f965b9698120f893a86f146efd05f7585659a3913f519`

Trong lần tái cấu trúc này:

- Không huấn luyện lại mô hình.
- Không tạo lại dữ liệu.
- Không ghi đè checkpoint.
- Không tái tạo `metrics.csv`.
- Không sinh lại `train_financial.py`.

Quy trình huấn luyện thực tế vẫn được bảo toàn trong các hàm và script riêng.

In [6]:
import ast
import json
import random
import hashlib

from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim


# 1. Tìm thư mục gốc repository
project_root = next(
    (
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (
            p / "experiments" / "03_financial"
            / "02_financial_model.ipynb"
        ).is_file()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError(
        "Không tìm thấy thư mục Financial."
    )


# 2. Đường dẫn dùng chung
experiment_dir = (
    project_root / "experiments" / "03_financial"
)

MODEL_NOTEBOOK = (
    experiment_dir / "02_financial_model.ipynb"
)

DATA_PATH = (
    experiment_dir / "data" / "financial_train_test.pt"
)

RESULTS_DIR = (
    experiment_dir
    / "results"
    / "financial_grid_search_e4000"
)


# 3. Chế độ bảo vệ kết quả
RUN_TRAINING = False


# 4. Thiết lập seed
SEED = 7

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# 5. Danh sách kiến trúc cần nạp
required_classes = {
    "StrictStableKoopmanOperator",
    "SinCosPosEnc",
    "PatchEmbed1D",
    "PatchTST_Backbone",
    "Koopformer_PatchTST",
    "SeriesDecomp",
    "SimpleAutoformer",
    "Koopformer_Autoformer",
    "InformerSparse",
    "Koopformer_Informer",
    "SimpleLSTMForecaster",
}

required_functions = {
    "orthogonalize",
    "compute_loss",
    "create_windows",
    "train_model",
    "evaluate_model",
}


# 6. Đọc notebook Model dưới dạng JSON
with open(MODEL_NOTEBOOK, "r", encoding="utf-8") as f:
    notebook = json.load(f)


# 7. Chỉ lấy các định nghĩa class và function cần thiết
definitions = []

for cell in notebook["cells"]:
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    if not source.strip():
        continue

    tree = ast.parse(source)

    for node in tree.body:
        if (
            isinstance(node, ast.ClassDef)
            and node.name in required_classes
        ) or (
            isinstance(node, ast.FunctionDef)
            and node.name in required_functions
        ):
            definitions.append(node)


# 8. Nạp định nghĩa, không chạy Training
module = ast.Module(
    body=definitions,
    type_ignores=[]
)

ast.fix_missing_locations(module)

exec(
    compile(module, str(MODEL_NOTEBOOK), "exec"),
    globals()
)


# 9. Kiểm tra đầy đủ định nghĩa
missing = sorted(
    name
    for name in required_classes | required_functions
    if name not in globals()
)

assert not missing, (
    f"Missing model definitions: {missing}"
)


# 10. Tổng kết
print("=== FINANCIAL MODEL LOADER ===")
print("Model notebook:", MODEL_NOTEBOOK)
print("Device:", DEVICE)
print("Seed:", SEED)

print("Imported classes:", len(required_classes))
print("Imported functions:", len(required_functions))
print("Training enabled:", RUN_TRAINING)

print("\nModel definitions loaded successfully.")

=== FINANCIAL MODEL LOADER ===
Model notebook: c:\Users\Win 11 New\Desktop\pa gk\experiments\03_financial\02_financial_model.ipynb
Device: cuda
Seed: 7
Imported classes: 11
Imported functions: 5
Training enabled: False

Model definitions loaded successfully.



## 2. Dữ liệu đầu vào và cấu hình thực nghiệm

### 2.1. Dữ liệu Financial

Thực nghiệm sử dụng hai segment Bitcoin đã được chuẩn hóa bằng Min-Max.

Tệp đầu vào:

`data/financial_train_test.pt`

Kích thước:

| Tập | Shape |
|---|---|
| Train | (2320, 2) |
| Test | (580, 2) |

Hai kênh tương ứng `BTC_segment_1` và `BTC_segment_2`.

Scaler được khớp trên Train và áp dụng cho cả Train và Test.

### 2.2. Grid Search

Không gian siêu tham số:

- Patch lengths: 40, 50, 60, 70, 80, 90, 100.
- Forecast horizons: 2, 4, 6, 8, 10, 12, 14, 16.
- Models: LSTM, PatchTST, Autoformer, Informer.

Tổng số lượt huấn luyện:

\[
7\times8\times4=224
\]

### 2.3. Cấu hình chung

| Tham số | Giá trị |
|---|---|
| `d_model` | 48 |
| Epochs | 4000 |
| Learning rate | 0.0003 |
| Lyapunov weight | 0.1 |
| Seed | 7 |
| Optimizer | Adam |

Các biến thể Koopman sử dụng chính quy hóa Lyapunov. Baseline LSTM được huấn luyện với hệ số Lyapunov bằng 0.

### 2.4. Tính toàn vẹn dữ liệu

Trước khi sử dụng dữ liệu, notebook xác minh:

- Hai kênh BTC.
- Kích thước Train/Test.
- Fingerprint SHA-256.
- Tính nhất quán giữa fingerprint lưu trong bundle và tensor thực tế.

Không tái tạo dữ liệu trong notebook này.


In [7]:
# 1. Đọc bundle dữ liệu đã chuẩn bị
if not DATA_PATH.is_file():
    raise FileNotFoundError(DATA_PATH)

# Bundle thuộc repository, có metadata NumPy.
bundle = torch.load(
    DATA_PATH,
    map_location="cpu",
    weights_only=False
)

train_data = bundle["train"]
test_data = bundle["test"]


# 2. Cấu hình dữ liệu
INPUT_DIM = 2

EXPECTED_CHANNELS = [
    "BTC_segment_1",
    "BTC_segment_2"
]

EXPECTED_HASH = (
    "9383a90ba5109d569c2f965b9698120f893a86"
    "f146efd05f7585659a3913f519"
)


# 3. Kiểm tra metadata
assert bundle["channels"] == EXPECTED_CHANNELS
assert bundle["input_dim"] == INPUT_DIM

assert train_data.shape == (2320, 2)
assert test_data.shape == (580, 2)

assert bundle["data_hash"] == EXPECTED_HASH


# 4. Tính lại fingerprint từ dữ liệu hiện có
actual_hash = hashlib.sha256(
    train_data.numpy().tobytes()
    + test_data.numpy().tobytes()
).hexdigest()

assert actual_hash == EXPECTED_HASH, (
    "Financial data differs from trained checkpoints!"
)


# 5. Grid Search
PATCH_LENGTHS = [40, 50, 60, 70, 80, 90, 100]

HORIZONS = [2, 4, 6, 8, 10, 12, 14, 16]

MODELS = [
    "LSTM",
    "PatchTST",
    "Autoformer",
    "Informer"
]


# 6. Training hyperparameters
D_MODEL = 48
EPOCHS = 4000
LEARNING_RATE = 3e-4
LYAP_WEIGHT = 0.1

TOTAL_RUNS = (
    len(PATCH_LENGTHS)
    * len(HORIZONS)
    * len(MODELS)
)

assert TOTAL_RUNS == 224


# 7. Tổng kết
print("=== FINANCIAL TRAINING CONFIGURATION ===")
print("Train:", tuple(train_data.shape))
print("Test:", tuple(test_data.shape))
print("Channels:", bundle["channels"])

print("\nModels:", MODELS)
print("Patch lengths:", PATCH_LENGTHS)
print("Forecast horizons:", HORIZONS)

print("\nTotal configurations:", TOTAL_RUNS)
print("Epochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Lyapunov weight:", LYAP_WEIGHT)

print("\nDataset SHA256:", actual_hash)
print("Hash verified:", True)

=== FINANCIAL TRAINING CONFIGURATION ===
Train: (2320, 2)
Test: (580, 2)
Channels: ['BTC_segment_1', 'BTC_segment_2']

Models: ['LSTM', 'PatchTST', 'Autoformer', 'Informer']
Patch lengths: [40, 50, 60, 70, 80, 90, 100]
Forecast horizons: [2, 4, 6, 8, 10, 12, 14, 16]

Total configurations: 224
Epochs: 4000
Learning rate: 0.0003
Lyapunov weight: 0.1

Dataset SHA256: 9383a90ba5109d569c2f965b9698120f893a86f146efd05f7585659a3913f519
Hash verified: True



## 3. Factory khởi tạo mô hình

### 3.1. Mục đích

Hàm `create_model()` khởi tạo một trong bốn kiến trúc theo tên mô hình và cấu hình dự báo.

Các tham số:

- `model_name`: tên mô hình.
- `patch_len`: độ dài chuỗi đầu vào và patch length.
- `horizon`: số bước dự báo.
- `d_model`: kích thước biểu diễn mô hình.

### 3.2. Các mô hình

**LSTM:** Hai lớp LSTM với hidden dimension bằng `d_model`.

**PatchTST:** PatchTST backbone kết hợp toán tử Koopman ổn định.

**Autoformer:** Backbone Autoformer đơn giản hóa kết hợp toán tử Koopman.

**Informer:** Backbone Informer đơn giản hóa kết hợp toán tử Koopman.

### 3.3. Kích thước đầu ra

Với hai kênh Financial, tất cả mô hình đều tạo dự báo có dạng:

\[
\hat{Y}\in\mathbb{R}^{B\times2H}
\]

### 3.4. Tương thích checkpoint

Factory giữ nguyên kiến trúc, tham số và tên các lớp đã dùng khi huấn luyện.

Notebook không thực hiện smoke test ở phần này nhằm tránh tạo những mô hình ngẫu nhiên không cần thiết.


In [8]:
def create_model(model_name, patch_len, horizon, d_model=48):

    if model_name == "LSTM":
        model = SimpleLSTMForecaster(
            input_dim=INPUT_DIM,
            hidden_dim=d_model,
            num_layers=2,
            horizon=horizon
        )

    elif model_name == "PatchTST":
        model = Koopformer_PatchTST(
            input_dim=INPUT_DIM,
            seq_len=patch_len,
            horizon=horizon,
            patch_len=patch_len,
            d_model=d_model
        )

    elif model_name == "Autoformer":
        model = Koopformer_Autoformer(
            input_dim=INPUT_DIM,
            seq_len=patch_len,
            horizon=horizon,
            patch_len=patch_len,
            d_model=d_model
        )

    elif model_name == "Informer":
        model = Koopformer_Informer(
            input_dim=INPUT_DIM,
            seq_len=patch_len,
            horizon=horizon,
            patch_len=patch_len,
            d_model=d_model
        )

    else:
        raise ValueError(f"Unknown model: {model_name}")

    return model.to(DEVICE)


## 4. Huấn luyện một cấu hình và lưu checkpoint

### 4.1. Mục tiêu

Hàm `run_one_configuration()` huấn luyện một mô hình tại một cặp patch length và forecast horizon.

Đầu vào gồm:

- `model_name`
- `patch_len`
- `horizon`

### 4.2. Quy trình

1. Xác định đường dẫn checkpoint.
2. Nếu checkpoint đã tồn tại, kiểm tra metadata và bỏ qua lượt huấn luyện.
3. Tạo sliding windows cho Train và Test.
4. Khởi tạo mô hình.
5. Chọn hệ số chính quy hóa Lyapunov.
6. Huấn luyện bằng `train_model()`.
7. Đánh giá Train và Test.
8. Lưu trọng số, dự báo, nhãn, metric và lịch sử huấn luyện.
9. Giải phóng các biến và bộ nhớ GPU sau khi hoàn thành.

### 4.3. Quy tắc tạo sliding windows

Sử dụng `create_windows()` từ notebook Model.

Hàm giữ nguyên:

`range(len(data) - seq_len - horizon)`

Nhãn được làm phẳng theo thứ tự channel-major.

### 4.4. Chính quy hóa

Đối với LSTM:

\[
\lambda_{Lyap}=0
\]

Đối với các biến thể Koopman:

\[
\lambda_{Lyap}=0.1
\]

### 4.5. Cấu trúc checkpoint

Mỗi checkpoint chứa:

- `model_state_dict`: trọng số mô hình cuối cùng.
- `pred_train`, `truth_train`: dự báo và nhãn Train.
- `pred_test`, `truth_test`: dự báo và nhãn Test.
- `loss`, `mse_history`, `lyap_history`, `eig`: lịch sử huấn luyện.
- `train_mse`, `train_mae`, `test_mse`, `test_mae`.
- Siêu tham số và tên mô hình.
- Tên các kênh và SHA-256 fingerprint của dữ liệu.

### 4.6. Cơ chế checkpoint

Nếu checkpoint hoàn chỉnh đã tồn tại, hàm kiểm tra metadata rồi bỏ qua lượt huấn luyện.

Cơ chế này **không phải resume từ một epoch bất kỳ**: nó không lưu trạng thái Adam optimizer để tiếp tục giữa chừng từ epoch bị gián đoạn.

Checkpoint được ghi bằng tệp tạm rồi đổi tên khi quá trình lưu hoàn tất.

### 4.7. Bảo toàn kết quả

Hàm được giữ để tái lập thực nghiệm nhưng không được gọi trong quá trình cleanup.


In [9]:
def run_one_configuration(model_name, patch_len, horizon):

    # 1. Checkpoint directory
    results_dir = (
        DATA_PATH.parent.parent
        / "results"
        / "financial_grid_search_e4000"
    )
    results_dir.mkdir(parents=True, exist_ok=True)

    checkpoint_path = results_dir / (
        f"{model_name.lower()}_p{patch_len}"
        f"_h{horizon}_d{D_MODEL}.pt"
    )

    # 2. Skip configurations already completed
    if checkpoint_path.exists():

        saved = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=False
        )

        assert saved["data_hash"] == bundle["data_hash"]
        assert saved["epochs"] == EPOCHS
        assert saved["model"] == model_name
        assert saved["patch_len"] == patch_len
        assert saved["horizon"] == horizon
        assert saved["d_model"] == D_MODEL

        print(f"SKIP: {checkpoint_path.name}")

        return checkpoint_path

    print(
        f"\nSTART: {model_name} | "
        f"P={patch_len}, H={horizon}, "
        f"d_model={D_MODEL}"
    )

    # 3. Create sliding windows
    X_train, y_train = create_windows(
        train_data, patch_len, horizon
    )

    X_test, y_test = create_windows(
        test_data, patch_len, horizon
    )

    # 4. Initialize model
    model = create_model(
        model_name=model_name,
        patch_len=patch_len,
        horizon=horizon,
        d_model=D_MODEL
    )

    # 5. Select Lyapunov weight
    lyap_weight = (
        0.0 if model_name == "LSTM"
        else LYAP_WEIGHT
    )

    # 6. Train the model
    result = train_model(
        model=model,
        X_train=X_train,
        y_train=y_train,
        epochs=EPOCHS,
        lr=LEARNING_RATE,
        lyap_weight=lyap_weight,
        verbose=True
    )

    trained_model = result["model"]

    # 7. Evaluate Train and Test
    train_metrics = evaluate_model(
        trained_model, X_train, y_train
    )

    test_metrics = evaluate_model(
        trained_model, X_test, y_test
    )

    # 8. Build checkpoint
    checkpoint = {
        "model": model_name,

        "model_state_dict": {
            k: v.detach().cpu().clone()
            for k, v in trained_model.state_dict().items()
        },

        "pred_train": train_metrics["predictions"],
        "truth_train": y_train.cpu(),

        "pred_test": test_metrics["predictions"],
        "truth_test": y_test.cpu(),

        "loss": torch.tensor(result["loss"]),
        "eig": torch.tensor(result["eig"]),
        "mse_history": torch.tensor(result["mse"]),
        "lyap_history": torch.tensor(result["lyap"]),

        "train_mse": train_metrics["MSE"],
        "train_mae": train_metrics["MAE"],

        "test_mse": test_metrics["MSE"],
        "test_mae": test_metrics["MAE"],

        "patch_len": patch_len,
        "horizon": horizon,
        "d_model": D_MODEL,
        "input_dim": INPUT_DIM,

        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "lyap_weight": lyap_weight,

        "channels": bundle["channels"],
        "data_hash": bundle["data_hash"],
    }

    # 9. Atomic checkpoint save
    temp_path = checkpoint_path.with_suffix(".pt.tmp")

    torch.save(checkpoint, temp_path)
    temp_path.replace(checkpoint_path)

    print(
        f"DONE: {model_name} | "
        f"P={patch_len}, H={horizon}"
    )

    print(
        f"Train MSE: {train_metrics['MSE']:.6f} | "
        f"Test MSE: {test_metrics['MSE']:.6f}"
    )

    print(f"Saved: {checkpoint_path}")

    # 10. Release GPU memory after completed run
    del trained_model, model, result
    del X_train, y_train, X_test, y_test

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return checkpoint_path


print("Financial training function defined successfully.")

Financial training function defined successfully.



## 5. Grid Search và quản lý kết quả

### 5.1. Không gian tìm kiếm

Grid Search gồm:

- 7 patch lengths.
- 8 forecast horizons.
- 4 mô hình.

Tổng số:

\[
7\times8\times4=224
\]

### 5.2. Quy ước checkpoint

Mỗi cấu hình tương ứng một checkpoint có tên:

`{model}_p{patch_len}_h{horizon}_d48.pt`

Các checkpoint nằm trong:

`results/financial_grid_search_e4000/`

### 5.3. Hàm checkpoint_file()

Tạo đường dẫn checkpoint theo một cấu hình cụ thể.

### 5.4. Hàm collect_metrics()

Đọc các checkpoint hoàn chỉnh, kiểm tra metadata và tổng hợp hai dòng kết quả cho mỗi checkpoint:

- Một dòng Train.
- Một dòng Test.

Các cột trong `metrics.csv`:

`Model, PatchLen, Horizon, d_model, Set, Epochs, MSE, MAE`

Hàm hiện tại còn thực hiện **ghi lại `metrics.csv`** theo cơ chế ghi tệp tạm và thay thế tệp đích.

Vì vậy, không gọi `collect_metrics()` trong chế độ kiểm toán chỉ đọc.

### 5.5. Hàm run_grid_search()

Hàm duyệt toàn bộ tổ hợp cấu hình, bỏ qua checkpoint đã hoàn thành và huấn luyện các cấu hình còn thiếu.

Sau mỗi lượt huấn luyện, hàm gọi `collect_metrics()` để cập nhật CSV.

Tham số `max_new_runs` giới hạn số lượt huấn luyện mới.

### 5.6. Phạm vi sử dụng

Cả `collect_metrics()` và `run_grid_search()` được giữ nguyên để phục vụ tái lập thực nghiệm.

Trong quá trình cleanup, chỉ sử dụng cell kiểm toán kết quả riêng và không gọi hai hàm này.


In [10]:
from itertools import product
import os

RESULTS_DIR = (
    DATA_PATH.parent.parent
    / "results"
    / "financial_grid_search_e4000"
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

METRICS_PATH = RESULTS_DIR / "metrics.csv"

METRIC_COLUMNS = [
    "Model", "PatchLen", "Horizon",
    "d_model", "Set", "Epochs", "MSE", "MAE"
]

CONFIGURATIONS = list(
    product(PATCH_LENGTHS, HORIZONS, MODELS)
)


def checkpoint_file(model_name, patch_len, horizon):
    return RESULTS_DIR / (
        f"{model_name.lower()}_p{patch_len}"
        f"_h{horizon}_d{D_MODEL}.pt"
    )


def collect_metrics():
    """
    Rebuild metrics.csv from completed checkpoints.
    Checkpoints are the source of truth.
    """

    rows = []
    completed = 0

    for patch_len, horizon, model_name in CONFIGURATIONS:

        path = checkpoint_file(
            model_name, patch_len, horizon
        )

        if not path.exists():
            continue

        ckpt = torch.load(
            path,
            map_location="cpu",
            weights_only=False
        )

        # Verify checkpoint identity
        assert ckpt["data_hash"] == bundle["data_hash"]
        assert ckpt["epochs"] == EPOCHS
        assert ckpt["model"] == model_name
        assert ckpt["patch_len"] == patch_len
        assert ckpt["horizon"] == horizon
        assert ckpt["d_model"] == D_MODEL

        completed += 1

        for dataset in ["Train", "Test"]:

            prefix = dataset.lower()

            rows.append({
                "Model": model_name,
                "PatchLen": patch_len,
                "Horizon": horizon,
                "d_model": D_MODEL,
                "Set": dataset,
                "Epochs": EPOCHS,
                "MSE": ckpt[f"{prefix}_mse"],
                "MAE": ckpt[f"{prefix}_mae"]
            })

    metrics_df = pd.DataFrame(
        rows, columns=METRIC_COLUMNS
    )

    # Atomic CSV save
    temp_path = METRICS_PATH.with_suffix(".csv.tmp")

    metrics_df.to_csv(temp_path, index=False)
    os.replace(temp_path, METRICS_PATH)

    return completed, metrics_df


def run_grid_search(max_new_runs=None):

    completed, _ = collect_metrics()

    print(f"Total configurations: {len(CONFIGURATIONS)}")
    print(f"Already completed: {completed}")
    print(f"Epochs per configuration: {EPOCHS}")
    print(f"Device: {DEVICE}")

    new_runs = 0

    for index, (patch_len, horizon, model_name) in enumerate(
        CONFIGURATIONS, start=1
    ):

        path = checkpoint_file(
            model_name, patch_len, horizon
        )

        if path.exists():
            print(f"SKIP [{index}/224] {path.name}")
            continue

        if max_new_runs is not None and new_runs >= max_new_runs:
            break

        print(
            f"\n[{index}/224] "
            f"{model_name} | P={patch_len}, H={horizon}"
        )

        # Train and save checkpoint
        run_one_configuration(
            model_name=model_name,
            patch_len=patch_len,
            horizon=horizon
        )

        new_runs += 1

        # Update CSV immediately after each completed run
        completed, metrics_df = collect_metrics()

        print(
            f"PROGRESS: {completed}/224 | "
            f"Metrics rows: {len(metrics_df)}"
        )

    completed, metrics_df = collect_metrics()

    print("\nGRID SEARCH STATUS")
    print("Completed:", completed, "/224")
    print("New runs:", new_runs)
    print("Metrics rows:", len(metrics_df))
    print("Results directory:", RESULTS_DIR)

    return metrics_df


print("Grid Search functions defined successfully.")
print("Total configurations:", len(CONFIGURATIONS))

Grid Search functions defined successfully.
Total configurations: 224



## 6. Xác minh kết quả huấn luyện đã lưu

### 6.1. Mục tiêu

Kiểm tra tính đầy đủ của Grid Search Financial mà không thực hiện huấn luyện hoặc ghi lại kết quả.

### 6.2. Kết quả kỳ vọng

| Hạng mục | Số lượng |
|---|---|
| Checkpoint | 224 |
| Dòng metrics | 448 |
| Dòng Train | 224 |
| Dòng Test | 224 |
| Models | 4 |
| Patch lengths | 7 |
| Horizons | 8 |

Mỗi tổ hợp `(Model, PatchLen, Horizon)` phải có đúng một checkpoint và hai dòng metric tương ứng với Train/Test.

### 6.3. Kiểm tra

Cell kiểm toán xác minh:

1. Tệp `metrics.csv` tồn tại.
2. Có đúng 224 checkpoint với tên tương ứng toàn bộ cấu hình.
3. Có đầy đủ 448 bản ghi metrics.
4. Không trùng cấu hình trong cùng tập Train hoặc Test.
5. Giá trị MSE/MAE hữu hạn.
6. Metadata cấu hình nhất quán.
7. Script `train_financial.py` vẫn tồn tại.

### 6.4. Tính toàn vẹn dữ liệu

Trước khi dọn notebook, 224 checkpoint đã được kiểm tra và đều khớp với fingerprint của bundle Financial.

Cell kiểm toán này chỉ kiểm tra cấu trúc kết quả, không tải lại toàn bộ 224 checkpoint hoặc tính lại trọng số.

### 6.5. Script GPU

Tệp `train_financial.py` là script huấn luyện đã được xuất để chạy trên GPU.

Notebook không sinh lại hoặc ghi đè script này trong quá trình cleanup.

### 6.6. Chế độ thực thi

`RUN_TRAINING = False`

Do đó, chạy toàn bộ notebook chỉ thực hiện các thao tác đọc, định nghĩa hàm và kiểm tra kết quả, không khởi động Grid Search.


In [11]:
# 1. Chỉ cho phép kiểm toán trong notebook này
assert RUN_TRAINING is False, (
    "Training mode must remain disabled during cleanup."
)


# 2. Đường dẫn kết quả đã lưu
METRICS_PATH = RESULTS_DIR / "metrics.csv"

TRAINING_SCRIPT = (
    experiment_dir / "train_financial.py"
)

if not RESULTS_DIR.is_dir():
    raise FileNotFoundError(RESULTS_DIR)

if not METRICS_PATH.is_file():
    raise FileNotFoundError(METRICS_PATH)

if not TRAINING_SCRIPT.is_file():
    raise FileNotFoundError(TRAINING_SCRIPT)


# 3. Đọc metrics (không ghi file)
saved_metrics = pd.read_csv(METRICS_PATH)

assert list(saved_metrics.columns) == METRIC_COLUMNS


# 4. Kiểm tra số lượng checkpoint
checkpoint_paths = sorted(
    RESULTS_DIR.glob("*.pt")
)

actual_checkpoint_names = {
    path.name for path in checkpoint_paths
}

expected_checkpoint_names = {
    checkpoint_file(model, p, h).name
    for p in PATCH_LENGTHS
    for h in HORIZONS
    for model in MODELS
}

assert len(checkpoint_paths) == 224

assert (
    actual_checkpoint_names
    == expected_checkpoint_names
), "Missing or unexpected checkpoints."


# 5. Kiểm tra đầy đủ 448 dòng metrics
assert len(saved_metrics) == 448

train_rows = saved_metrics[
    saved_metrics["Set"] == "Train"
]

test_rows = saved_metrics[
    saved_metrics["Set"] == "Test"
]

assert len(train_rows) == 224
assert len(test_rows) == 224


# 6. Kiểm tra mỗi cấu hình xuất hiện đúng một lần
expected_keys = {
    (model, p, h, D_MODEL, split)
    for model in MODELS
    for p in PATCH_LENGTHS
    for h in HORIZONS
    for split in ["Train", "Test"]
}

actual_keys = list(zip(
    saved_metrics["Model"],
    saved_metrics["PatchLen"],
    saved_metrics["Horizon"],
    saved_metrics["d_model"],
    saved_metrics["Set"]
))

assert len(actual_keys) == len(set(actual_keys)), (
    "Duplicate configurations in metrics.csv."
)

assert set(actual_keys) == expected_keys, (
    "Missing or unexpected metric configurations."
)


# 7. Kiểm tra epoch và metric
assert (
    saved_metrics["Epochs"] == EPOCHS
).all()

assert np.isfinite(
    saved_metrics[["MSE", "MAE"]]
    .to_numpy(dtype=float)
).all()


# 8. Kiểm tra fingerprint dữ liệu đã nạp
assert bundle["data_hash"] == EXPECTED_HASH


# 9. In kết quả kiểm toán
print("=== FINANCIAL GRID SEARCH AUDIT ===")

print("Results directory:", RESULTS_DIR)
print("Training script:", TRAINING_SCRIPT.name)

print("\nExpected configurations:", TOTAL_RUNS)
print("Checkpoints:", len(checkpoint_paths))

print("\nMetrics rows:", len(saved_metrics))
print("Train rows:", len(train_rows))
print("Test rows:", len(test_rows))

print("\nDataset SHA256:", bundle["data_hash"])
print("Training enabled:", RUN_TRAINING)

print("\nAll saved configurations are present.")
print("No training started.")
print("No checkpoint or metrics file written.")

=== FINANCIAL GRID SEARCH AUDIT ===
Results directory: c:\Users\Win 11 New\Desktop\pa gk\experiments\03_financial\results\financial_grid_search_e4000
Training script: train_financial.py

Expected configurations: 224
Checkpoints: 224

Metrics rows: 448
Train rows: 224
Test rows: 224

Dataset SHA256: 9383a90ba5109d569c2f965b9698120f893a86f146efd05f7585659a3913f519
Training enabled: False

All saved configurations are present.
No training started.
No checkpoint or metrics file written.


## 6. Điều khiển thực thi Grid Search

### 6.1. Mục đích

Hàm `run_financial_experiment()` cung cấp điểm vào để thực hiện Grid Search trực tiếp từ notebook, thay thế cách chạy script `train_financial.py`.

Hàm sử dụng lại các định nghĩa đã có:

- `create_model()`
- `run_one_configuration()`
- `checkpoint_file()`
- `collect_metrics()`
- `run_grid_search()`

### 6.2. Cơ chế thực thi

Trước khi bắt đầu huấn luyện, hàm kiểm tra số checkpoint còn thiếu.

Nếu toàn bộ 224 checkpoint đã tồn tại, hàm chỉ đọc `metrics.csv` và không chạy lại huấn luyện.

Nếu còn cấu hình chưa hoàn thành, hàm chỉ huấn luyện khi người dùng chủ động bật `RUN_TRAINING` và xác nhận.

### 6.3. Giới hạn số lượt huấn luyện

Tham số `max_new_runs` quy định số cấu hình mới tối đa được huấn luyện trong một lần gọi.

Mặc định: `max_new_runs = 1`, tương ứng với `--limit 1` của script GPU trước đây.

### 6.4. Bảo toàn kết quả

Trong quá trình cleanup:

`RUN_TRAINING = False`

Notebook không thực hiện huấn luyện, không tạo checkpoint mới và không ghi lại `metrics.csv`.

Việc chạy Grid Search chỉ được kích hoạt khi có nhu cầu tái lập hoặc hoàn thiện các cấu hình còn thiếu.

In [ ]:
def run_financial_experiment(
    max_new_runs=1,
    confirm=False
):
    # 1. Kiểm tra giới hạn huấn luyện
    if not isinstance(max_new_runs, int) or max_new_runs < 1:
        raise ValueError(
            "max_new_runs phải là số nguyên >= 1."
        )

    # 2. Lấy toàn bộ cấu hình đã định nghĩa
    expected_paths = [
        checkpoint_file(model, p, h)
        for p in PATCH_LENGTHS
        for h in HORIZONS
        for model in MODELS
    ]

    completed_paths = [
        path for path in expected_paths
        if path.is_file()
    ]

    missing_paths = [
        path for path in expected_paths
        if not path.is_file()
    ]

    print("=== FINANCIAL GRID SEARCH ===")
    print("Device:", DEVICE)
    print("Total configurations:", len(expected_paths))
    print("Completed:", len(completed_paths))
    print("Missing:", len(missing_paths))
    print("Epochs:", EPOCHS)
    print("Max new runs:", max_new_runs)
    print("Data hash:", bundle["data_hash"])

    # 3. Nếu tất cả checkpoint đã có:
    #    chỉ đọc kết quả, không ghi file
    if not missing_paths:
        print("\nAll 224 checkpoints already exist.")
        print("No training required.")

        if not METRICS_PATH.is_file():
            raise FileNotFoundError(METRICS_PATH)

        return pd.read_csv(METRICS_PATH)

    # 4. Không được huấn luyện nếu chưa bật chế độ
    if not RUN_TRAINING or not confirm:
        print("\nTraining disabled.")
        print("Existing checkpoints remain unchanged.")

        return None

    # 5. Chỉ chạy khi người dùng chủ động xác nhận
    print("\nStarting Financial Grid Search...")

    metrics_df = run_grid_search(
        max_new_runs=max_new_runs
    )

    print("\nGrid Search execution finished.")

    return metrics_df


print("Financial Grid Search entry point ready.")
print("Training enabled:", RUN_TRAINING)

In [ ]:
# 7. EXECUTION CONTROL

# Giữ False khi kiểm tra hoặc dọn repository.
# Chỉ chuyển True nếu chủ động muốn huấn luyện bổ sung.
RUN_TRAINING = False

# Tương đương --limit 1 của script cũ
MAX_NEW_RUNS = 1


if RUN_TRAINING:
    financial_metrics = run_financial_experiment(
        max_new_runs=MAX_NEW_RUNS,
        confirm=True
    )
else:
    print("Training is disabled.")
    print("No training or checkpoint modifications.")